In [1]:
import psycopg2
from decimal import Decimal
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(".env.local")

True

In [3]:
DATABASE_URL = os.getenv("DATABASE")

In [4]:
try:
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()

    cur.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """)

    tables = cur.fetchall()

    print("Tables in the database:")
    print("-" * 25)
    for table in tables:
        print(table[0])

    cur.close()
    conn.close()

except Exception as e:
    print(f"Error: {e}")

Tables in the database:
-------------------------
appointments
patients
post_consultation_instructions
service_recommendations
services


In [7]:
try:
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()
    
    cur.execute('SELECT * FROM services;')
    
    patients = cur.fetchall()
    
    colnames = [desc[0] for desc in cur.description]
    print(f"Columns: {colnames}")
    print("-" * 60)

    for patient in patients:
        print(patient)

    cur.close()
    conn.close()
    
except Exception as e:
    print(f"Unable to connect to the database: {e}")

Columns: ['id', 'name', 'description', 'price', 'duration_minutes', 'modality', 'is_active', 'created_at']
------------------------------------------------------------
(1, 'COVID-19 Initial Triage', 'Chat-based assessment of symptoms (headache, congestion) to determine risk level.', Decimal('0.00'), 15, 'Virtual', True, datetime.datetime(2026, 4, 20, 4, 0, 43, 980972))
(2, 'Specialist Teleconsultation', 'Virtual meeting with a doctor to discuss COVID-19 symptoms and medical history.', Decimal('55.00'), 20, 'Virtual', True, datetime.datetime(2026, 4, 20, 4, 0, 43, 980972))
(3, 'In-Clinic Antigen Test', 'Onsite rapid protein test with results delivered via email within 30 minutes.', Decimal('30.00'), 15, 'Onsite', True, datetime.datetime(2026, 4, 20, 4, 0, 43, 980972))
(4, 'Home PCR Collection', 'Onsite visit by a nurse to collect samples for laboratory PCR testing.', Decimal('120.00'), 30, 'Onsite', True, datetime.datetime(2026, 4, 20, 4, 0, 43, 980972))
(5, 'Automated Follow-up Care', 

In [6]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

services_to_insert = [
    (
        'COVID-19 Initial Triage', 
        'Chat-based assessment of symptoms (headache, congestion) to determine risk level.', 
        Decimal('0.00'), 15, 'Virtual'
    ),
    (
        'Specialist Teleconsultation', 
        'Virtual meeting with a doctor to discuss COVID-19 symptoms and medical history.', 
        Decimal('55.00'), 20, 'Virtual'
    ),
    (
        'In-Clinic Antigen Test', 
        'Onsite rapid protein test with results delivered via email within 30 minutes.', 
        Decimal('30.00'), 15, 'Onsite'
    ),
    (
        'Home PCR Collection', 
        'Onsite visit by a nurse to collect samples for laboratory PCR testing.', 
        Decimal('120.00'), 30, 'Onsite'
    ),
    (
        'Automated Follow-up Care', 
        '7-day monitoring system with daily email notifications and symptom tracking.', 
        Decimal('15.00'), 5, 'Virtual'
    )
]

insert_query = """
    INSERT INTO services (name, description, price, duration_minutes, modality)
    VALUES (%s, %s, %s, %s, %s);
"""

try:
    cur.executemany(insert_query, services_to_insert)
    
    conn.commit()
    print(f"Successfully added {len(services_to_insert)} healthcare services.")

except psycopg2.Error as e:
    conn.rollback()
    print(f"Database error: {e}")

finally:
    cur.close()
    conn.close()

Successfully added 5 healthcare services.
